In [11]:
import torch
import math
import numpy as np
import torch.nn as nn
import torch.nn.functional as f

## Embeddings

In [12]:
class Embeddings(nn.Module):
  def __init__(self, vocab_size, d_model):
    super().__init__()
    self.lut = nn.Embedding(vocab_size, d_model,padding_idx=0)
    self.d_model = d_model

  def forward(self, x):
    return self.lut(x) * math.sqrt(self.d_model)

## Positional Encoder

In [13]:
class PositionalEncoding(nn.Module):
  def __init__(self, d_model, dropout, max_len=5000):
    super().__init__()
    self.dropout = nn.Dropout(dropout)

    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0,max_len).unsqueeze(1)
    div_term = torch.exp(torch.arange(0,d_model,2) * -(math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    pe = pe.unsqueeze(0)
    self.register_buffer('pe', pe)

  def forward(self, x):
    x = x + self.pe[:, :x.size(1)].requires_grad_(False)
    return self.dropout(x)

## Attention

In [14]:
def attention(query, key, value, mask=None, dropout=None):
  d_k = query.size(-1)
  scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
  if mask is not None:
    scores = scores.masked_fill(mask == 0, -1e9)
  p_attn = scores.softmax(dim=-1)
  if dropout is not None:
    p_attn = dropout(p_attn)
  return torch.matmul(p_attn, value), p_attn

## Positionwise Feedforward neural network

In [15]:
class PositionwiseFeedForward(nn.Module):
  def __init__(self, d_model, d_ff, dropout=0.1):
    super().__init__()
    self.w_1 = nn.Linear(d_model, d_ff)
    self.w_2 = nn.Linear(d_ff, d_model)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    return self.w_2(self.dropout(f.relu(self.w_1(x))))

## Layer Normalization

In [16]:
class LayerNorm(nn.Module):
  def __init__(self,features,eps = 1e-6):
    super().__init__()
    self.a_2 = nn.Parameter(torch.ones(features))
    self.b_2 = nn.Parameter(torch.zeros(features))
    self.eps = eps

  def forward(self, x):
    mean = x.mean(-1, keepdim=True)
    std = x.std(-1, keepdim=True)
    return self.a_2 * (x - mean) / (std + self.eps) + self.b_2

## Sublayer Connection

In [17]:
class SublayerConnection(nn.Module):
    def __init__(self, size, dropout):
        super().__init__()
        self.norm = LayerNorm(size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, sublayer):
          return x + self.dropout(
              sublayer(
                  self.norm(x)
              )
          )

## Clone

In [18]:
import copy

def clone(module,cnt):
  return nn.ModuleList([copy.deepcopy(module) for _ in range(cnt)])

## Encoder Layer

In [19]:
class EncoderLayer(nn.Module):
  def __init__(self, size, self_attn, feed_forward, dropout):
    super().__init__()
    self.self_attn = self_attn
    self.feed_forward = feed_forward
    self.sublayer = clone(SublayerConnection(size, dropout), 2)
    self.size = size

  def forward(self,x,mask):
    x = self.sublayer[0](x, lambda x: self.self_attn(x,x,x,mask))
    return self.sublayer[1](x, self.feed_forward)

## Encoder Block

In [20]:
class Encoder(nn.Module):
  def __init__(self,layer,N):
    super().__init__()
    self.layers = clone(layer,N)
    self.norm = LayerNorm(layer.size)

  def forward(self,x,mask):
    for layer in self.layers:
      x = layer(x,mask)
    return self.norm(x)

## Decoder Layer

In [21]:
class DecoderLayer(nn.Module):
    def __init__(self, size, self_attn, src_attn, feed_forward, dropout):
        super().__init__()
        self.size = size
        self.self_attn = self_attn
        self.src_attn = src_attn
        self.feed_forward = feed_forward
        self.sublayer = clone(SublayerConnection(size, dropout), 3)

    def forward(self, x, memory, src_mask, tgt_mask):
        m = memory
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, tgt_mask))
        x = self.sublayer[1](x, lambda x: self.src_attn(x, m, m, src_mask))
        return self.sublayer[2](x, self.feed_forward)

## Decoder Block

In [22]:
class Decoder(nn.Module):
    def __init__(self, layer, N):
        super().__init__()
        self.layers = clone(layer, N)
        self.norm = LayerNorm(layer.size)

    def forward(self, x, memory, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, memory, src_mask, tgt_mask)
        return self.norm(x)

## Encoder Decoder Block

In [23]:
class EncoderDecoder(nn.Module):
  def __init__(self, encoder, decoder, src_embed, tgt_embed, generator):
    super().__init__()
    self.encoder = encoder
    self.decoder = decoder
    self.src_embed = src_embed
    self.tgt_embed = tgt_embed
    self.generator = generator

  def forward(self, src, tgt, src_mask, tgt_mask):
    return self.decode(self.encode(src,src_mask),src_mask,tgt,tgt_mask)

  def encode(self, src, src_mask):
    return self.encoder(self.src_embed(src),src_mask)

  def decode(self, memory, src_mask, tgt, tgt_mask):
    return self.decoder(self.tgt_embed(tgt),memory,src_mask,tgt_mask)

## Generator

In [24]:
class Generator(nn.Module):
  def __init__(self, d_model, vocab):
    super().__init__()
    self.proj = nn.Linear(d_model, vocab)

  def forward(self,x):
    return f.log_softmax(self.proj(x),dim = -1)

## Subsequent Mask

In [25]:
def subsequent_mask(size):
    attn_shape = (1, size, size)
    subsequent_mask = torch.triu(torch.ones(attn_shape), diagonal=1).type(
        torch.uint8
    )
    return subsequent_mask == 0

## multihead Attention

In [26]:
class MultiHeadedAttention(nn.Module):
    def __init__(self, h, d_model, dropout=0.1):
        super().__init__()
        assert d_model % h == 0
        self.d_k = d_model // h
        self.h = h
        self.linears = clone(nn.Linear(d_model, d_model), 4)
        self.attn = None
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, query, key, value, mask=None):
        if mask is not None:
            mask = mask.unsqueeze(1)
        nbatches = query.size(0)

        query, key, value = [
            lin(x).view(nbatches, -1, self.h, self.d_k).transpose(1, 2)
            for lin, x in zip(self.linears, (query, key, value))
        ]

        x, self.attn = attention(
            query, key, value, mask=mask, dropout=self.dropout
        )

        x = (
            x.transpose(1, 2)
            .contiguous()
            .view(nbatches, -1, self.h * self.d_k)
        )
        del query
        del key
        del value
        return self.linears[-1](x)

## Model Creation

In [27]:
def make_model(
    src_vocab, tgt_vocab, N=3, d_model=256, d_ff=512, h=8, dropout=0.1
):
    c = copy.deepcopy
    attn = MultiHeadedAttention(h, d_model)
    ff = PositionwiseFeedForward(d_model, d_ff, dropout)
    position = PositionalEncoding(d_model, dropout)
    model = EncoderDecoder(
        Encoder(EncoderLayer(d_model, c(attn), c(ff), dropout), N),
        Decoder(DecoderLayer(d_model, c(attn), c(attn), c(ff), dropout), N),
        nn.Sequential(Embeddings(src_vocab, d_model), c(position)),
        nn.Sequential(Embeddings(tgt_vocab, d_model), c(position)),
        Generator(d_model, tgt_vocab),
    )

    for p in model.parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform_(p)
    return model

## Input setup

In [29]:
!pip install -q datasets spacy
!python -m spacy download en_core_web_sm -q
!python -m spacy download de_core_news_sm -q
print("Done installing!")


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
Done installing!



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [30]:
from datasets import load_dataset

dataset = load_dataset("bentrevett/multi30k")

# Split into train / validation / test
train_data = dataset["train"]
val_data   = dataset["validation"]
test_data  = dataset["test"]

print("Train sentences :", len(train_data))
print("Val sentences   :", len(val_data))
print("Test sentences  :", len(test_data))

# Let's look at a few examples
print("\nExample sentence pairs:")
for i in range(3):
    print(f"  English : {train_data[i]['en']}")
    print(f"  German  : {train_data[i]['de']}")
    print()

README.md:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

c:\Users\khail\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\khail\.cache\huggingface\hub\datasets--bentrevett--multi30k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


train.jsonl:   0%|          | 0.00/4.60M [00:00<?, ?B/s]

val.jsonl:   0%|          | 0.00/164k [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/156k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/29000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1014 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Train sentences : 29000
Val sentences   : 1014
Test sentences  : 1000

Example sentence pairs:
  English : Two young, White males are outside near many bushes.
  German  : Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.

  English : Several men in hard hats are operating a giant pulley system.
  German  : Mehrere Männer mit Schutzhelmen bedienen ein Antriebsradsystem.

  English : A little girl climbing into a wooden playhouse.
  German  : Ein kleines Mädchen klettert in ein Spielhaus aus Holz.



In [31]:
import spacy

nlp_en = spacy.load("en_core_web_sm")  # English tokenizer
nlp_de = spacy.load("de_core_news_sm") # German tokenizer

def tokenize_en(text):
    return [token.text.lower() for token in nlp_en.tokenizer(text)]

def tokenize_de(text):
    return [token.text.lower() for token in nlp_de.tokenizer(text)]

# Test it
print(tokenize_en("I love playing football!"))
print(tokenize_de("Ich liebe Fußball!"))

['i', 'love', 'playing', 'football', '!']
['ich', 'liebe', 'fußball', '!']


In [32]:
from collections import Counter

PAD, UNK, BOS, EOS = 0, 1, 2, 3

def build_vocab(sentences, tokenizer, min_freq=2):
    counter = Counter()
    for sent in sentences:
        counter.update(tokenizer(sent))

    vocab = {"<pad>": 0, "<unk>": 1, "<bos>": 2, "<eos>": 3}
    for word, count in counter.items():
        if count >= min_freq and word not in vocab:
            vocab[word] = len(vocab)
    return vocab

vocab_en = build_vocab([x["en"] for x in train_data], tokenize_en)
vocab_de = build_vocab([x["de"] for x in train_data], tokenize_de)

inv_vocab_de = {v: k for k, v in vocab_de.items()}

print("English vocabulary size:", len(vocab_en))
print("German  vocabulary size:", len(vocab_de))
print("Example — 'the' maps to number:", vocab_en.get("the"))

English vocabulary size: 5893
German  vocabulary size: 7853
Example — 'the' maps to number: 41


In [33]:
def sentence_to_tensor(sentence, vocab, tokenizer):
    tokens = tokenizer(sentence)
    ids = [BOS] + [vocab.get(t, UNK) for t in tokens] + [EOS]
    return torch.tensor(ids, dtype=torch.long)

def tensor_to_sentence(tensor, inv_vocab):
    words = []
    for idx in tensor.tolist():
        if idx in (PAD, BOS, EOS):
            continue
        words.append(inv_vocab.get(idx, "<unk>"))
    return " ".join(words)

t = sentence_to_tensor("A dog is running.", vocab_en, tokenize_en)
print("Sentence as numbers:", t.tolist())

Sentence as numbers: [2, 21, 91, 34, 281, 14, 3]


In [23]:
from torch.utils.data import Dataset,DataLoader
from torch.nn.utils.rnn import pad_sequence

class Translation(Dataset):
  def __init__(self,data):
    self.pairs = [
        (sentence_to_tensor(x["en"],vocab_en,tokenize_en),sentence_to_tensor(x["de"],vocab_de,tokenize_de))
        for x in data
    ]
  def __len__(self):
    return len(self.pairs)
  def __getitem__(self,i):
    return self.pairs[i]

def collate_fn(batch):
    src, tgt = zip(*batch)
    src = pad_sequence(src, batch_first=True, padding_value=PAD)
    tgt = pad_sequence(tgt, batch_first=True, padding_value=PAD)
    return src, tgt

BATCH_SIZE = 64

train_loader = DataLoader(Translation(train_data), batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(Translation(val_data),   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(Translation(test_data),  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"Train batches: {len(train_loader)}")
print(f"Val   batches: {len(val_loader)}")

# Check one batch shape
src, tgt = next(iter(train_loader))
print(f"\nOne batch — src shape: {src.shape}  (batch_size x seq_len)")
print(f"One batch — tgt shape: {tgt.shape}")

Train batches: 454
Val   batches: 16

One batch — src shape: torch.Size([64, 26])  (batch_size x seq_len)
One batch — tgt shape: torch.Size([64, 25])


In [35]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

model = make_model(len(vocab_en), len(vocab_de)).to(DEVICE)
params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model built! Total parameters: {params:,}")

Using device: cpu
Model built! Total parameters: 9,491,885


In [25]:
loss_ce = nn.CrossEntropyLoss(ignore_index=PAD)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

def make_masks(src, tgt):
    src_mask = (src != PAD).unsqueeze(-2)
    tgt_mask = (tgt != PAD).unsqueeze(-2)
    tgt_mask = tgt_mask & subsequent_mask(tgt.size(-1)).to(tgt.device)
    return src_mask, tgt_mask

def train_one_epoch(loader):
    model.train()
    total_loss = 0
    for batch_idx, (src, tgt) in enumerate(loader):
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)
        tgt_in  = tgt[:, :-1]
        tgt_out = tgt[:, 1:]

        src_mask, tgt_mask = make_masks(src, tgt_in)

        out  = model(src, tgt_in, src_mask, tgt_mask)
        out  = model.generator(out)

        B, T, V = out.shape
        loss = loss_ce(out.reshape(B * T, V), tgt_out.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

def evaluate(loader):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for src, tgt in loader:
            src, tgt = src.to(DEVICE), tgt.to(DEVICE)
            tgt_in  = tgt[:, :-1]
            tgt_out = tgt[:, 1:]
            src_mask, tgt_mask = make_masks(src, tgt_in)
            out  = model(src, tgt_in, src_mask, tgt_mask)
            out  = model.generator(out)
            B, T, V = out.shape
            loss = loss_ce(out.reshape(B * T, V), tgt_out.reshape(-1))
            total_loss += loss.item()
    return total_loss / len(loader)

EPOCHS = 10
best_val_loss = float("inf")

print("Starting training...\n")
for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(train_loader)
    val_loss   = evaluate(val_loader)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_model.pt")
        saved = " ← best model saved"
    else:
        saved = ""

    print(f"Epoch {epoch:2d}/{EPOCHS}  |  train loss: {train_loss:.4f}  |  val loss: {val_loss:.4f}{saved}")

print("\nTraining complete!")


Starting training...

Epoch  1/10  |  train loss: 5.3806  |  val loss: 4.1568 ← best model saved
Epoch  2/10  |  train loss: 3.8626  |  val loss: 3.4032 ← best model saved
Epoch  3/10  |  train loss: 3.2992  |  val loss: 2.9371 ← best model saved
Epoch  4/10  |  train loss: 2.9137  |  val loss: 2.6349 ← best model saved
Epoch  5/10  |  train loss: 2.6181  |  val loss: 2.3942 ← best model saved
Epoch  6/10  |  train loss: 2.3787  |  val loss: 2.2074 ← best model saved
Epoch  7/10  |  train loss: 2.1826  |  val loss: 2.0713 ← best model saved
Epoch  8/10  |  train loss: 2.0182  |  val loss: 1.9578 ← best model saved
Epoch  9/10  |  train loss: 1.8802  |  val loss: 1.8684 ← best model saved
Epoch 10/10  |  train loss: 1.7563  |  val loss: 1.7998 ← best model saved

Training complete!


In [36]:
model.load_state_dict(torch.load("best_model.pt", map_location=DEVICE))
model.eval()

def translate(english):
    src      = sentence_to_tensor(english, vocab_en, tokenize_en).unsqueeze(0).to(DEVICE)
    src_mask = (src != PAD).unsqueeze(-2)

    with torch.no_grad():
        memory = model.encode(src, src_mask)               # encode English once
        output = torch.tensor([[BOS]], device=DEVICE)      # start with <bos>

        for _ in range(60):
            tgt_mask  = subsequent_mask(output.size(1)).to(DEVICE)
            dec_out   = model.decode(memory, src_mask, output, tgt_mask)
            prob      = model.generator(dec_out[:, -1])    # look at last position only
            next_word = prob.argmax(dim=-1, keepdim=True)  # pick best word
            output    = torch.cat([output, next_word], dim=1)
            if next_word.item() == EOS:
                break

    return tensor_to_sentence(output[0], inv_vocab_de)

# Try it!
examples = [
    "A dog is running in the park.",
    "Two children are playing soccer.",
    "A woman is reading a book.",
    "The cat is sitting on the table.",
    "A man is eating a sandwich.",
]
print("Translations:\n")
for sent in examples:
    print(f"  EN: {sent}")
    print(f"  DE: {translate(sent)}")
    print()


Translations:

  EN: A dog is running in the park.
  DE: ein hund rennt im park .

  EN: Two children are playing soccer.
  DE: zwei kinder spielen fußball .

  EN: A woman is reading a book.
  DE: eine frau liest ein buch .

  EN: The cat is sitting on the table.
  DE: die katze sitzt auf dem tisch .

  EN: A man is eating a sandwich.
  DE: ein mann isst ein <unk> .



In [37]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class Data(BaseModel):
  text: str

@app.get('/')
def home():
  return {'message':'Welcome to Eng to German Translator'}

@app.post("/translate")
def translator(data: Data):
  print("Received:", data.text)

  result = translate(data.text)

  print("Result:", result)

  return {"translation": result}

In [39]:
import threading
import uvicorn

def run():
    uvicorn.run(app, host="127.0.0.1", port=5001)

threading.Thread(target=run, daemon=True).start()

INFO:     Started server process [10320]
INFO:     Waiting for application startup.
INFO:     Application startup complete.


INFO:     Uvicorn running on http://127.0.0.1:5001 (Press CTRL+C to quit)


INFO:     127.0.0.1:49274 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:49274 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:56749 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:56749 - "GET /openapi.json HTTP/1.1" 200 OK
Received: Hello, Welcome to Germany
Result: ein <unk> <unk> <unk> .
INFO:     127.0.0.1:50332 - "POST /translate HTTP/1.1" 200 OK
Received: Hello, Goodmorning
Result: ein <unk> <unk> .
INFO:     127.0.0.1:64710 - "POST /translate HTTP/1.1" 200 OK
Received: Hello, Good morning
Result: ein <unk> <unk> .
INFO:     127.0.0.1:64309 - "POST /translate HTTP/1.1" 200 OK
